# 02b - Kitchen-sink selection + per-checkpoint calibration

Big feature pool (~100 candidates) → 4 selection methods in parallel →
consensus shortlist → XGBoost → **per-checkpoint isotonic calibration** →
**per-checkpoint threshold tuning** → final evaluation.

Replaces notebook 02 once verified (current 02 had `minute_out` leakage
and used pre-curated 25 features without verification of selection).

| Section | What |
|---------|------|
| A | Load + merge sources, remove leakage, add legit time feature |
| B | Match-level train/test split |
| C | Variance + correlation pre-filter |
| D | LASSO with C-sweep |
| E | Stability LASSO (bootstrap) |
| F | BorutaPy (XGBoost-based) |
| G | RFE-CV |
| H | Consensus shortlist (votes ≥ 2) |
| I | Train XGBoost on each shortlist + comparison |
| J | Best model → per-checkpoint isotonic calibration |
| K | Per-checkpoint threshold tuning |
| L | Final test-set evaluation |
| M | Comparison summary + persist |


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys
import warnings
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
"""Modelling stack."""
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, average_precision_score,
    brier_score_loss, log_loss, confusion_matrix, classification_report,
)
from sklearn.isotonic import IsotonicRegression
from sklearn.feature_selection import RFECV
from boruta import BorutaPy
print(f"xgboost: {xgb.__version__}")


xgboost: 3.2.0


## Section A - Load & merge candidate pool

Three sources of features:

1. `features/all_engineered_features.csv` - 74 features (cross + per-domain).
2. `features/players_quarters_final_feature_engineered.csv` - teammate's
   feature set with `ratio_*`, `minutes_active`, `formation_offensiveness`.
3. Main panel - base context: position, is_home, subbed, minute_in.

Merge all on `(player_appearance_id, checkpoint)`. Reconcile checkpoint
encoding (teammate uses continuous match-minute codes).


In [3]:
"""Paths."""
DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = PROJECT_ROOT / "features"
MODELS_DIR = PROJECT_ROOT / "models"
ART_DIR = MODELS_DIR / "kitchen_sink"
ART_DIR.mkdir(exist_ok=True)


In [4]:
"""Load all three sources."""
all_eng = pd.read_csv(FEATURES_DIR / "all_engineered_features.csv")
fe_teammate = pd.read_csv(FEATURES_DIR / "players_quarters_final_feature_engineered.csv")
main = pd.read_csv(DATA_DIR / "players_quarters_final.csv")

print(f"all_engineered: {all_eng.shape}")
print(f"fe_teammate:    {fe_teammate.shape}")
print(f"main:           {main.shape}")

# Reconcile teammate checkpoint encoding (continuous minute -> period-local)
CHECKPOINT_REMAP = {
    "H1_15": "H1_15", "H1_30": "H1_30", "H1_45": "H1_45",
    "H2_60": "H2_15", "H2_75": "H2_30", "H2_90": "H2_45",
}
fe_teammate = fe_teammate.copy()
fe_teammate["checkpoint"] = fe_teammate["checkpoint"].map(CHECKPOINT_REMAP)
print(f"after remap: {sorted(fe_teammate['checkpoint'].unique())}")


all_engineered: (3486, 74)
fe_teammate:    (3114, 44)
main:           (3486, 33)
after remap: ['H1_15', 'H1_30', 'H1_45', 'H2_15', 'H2_30', 'H2_45']


In [5]:
"""Merge."""
JOIN = ["player_appearance_id", "checkpoint"]

# Start from main (3486 rows) - but use INNER join with teammate to drop GK + ET1.
panel = main.merge(
    fe_teammate.drop(columns=[c for c in fe_teammate.columns
                               if c in main.columns and c not in JOIN]),
    on=JOIN, how="inner",
)
panel = panel.merge(
    all_eng.drop(columns=[c for c in all_eng.columns
                           if c in panel.columns and c not in JOIN]),
    on=JOIN, how="left",
)
print(f"merged panel: {panel.shape}")
print(f"target rate: {panel['scored_after'].mean():.4f}")


merged panel: (3114, 117)
target rate: 0.0649


In [6]:
"""Pre-process: remove leakage, add legit time, encode categoricals."""
# Drop leakage and replace with a leakage-free time feature.
panel["mins_on_pitch_so_far"] = (
    panel["checkpoint"].map({
        "H1_15": 15, "H1_30": 30, "H1_45": 45,
        "H2_15": 60, "H2_30": 75, "H2_45": 90, "ET1_15": 105,
    })
    - panel["minute_in"]
).clip(lower=0)

# Encode categoricals.
panel["is_home_int"] = panel["is_home"].astype(bool).astype(int)
panel["subbed_int"] = panel["subbed"].astype(str).str.upper().eq("TRUE").astype(int)
position_dummies = pd.get_dummies(panel["position"], prefix="pos").astype(int)
panel = pd.concat([panel, position_dummies], axis=1)

# Identify candidate features.
TARGET = "scored_after"
GROUP = "fixture_id"
ID_COLS = ["player_appearance_id", "player_id", "fixture_id", "date",
           "checkpoint", "checkpoint_period", "checkpoint_min"]
LEAKAGE_COLS = ["minute_out"]   # drop - post-hoc
DROP_COLS = ID_COLS + LEAKAGE_COLS + ["position", "formation", "is_home", "subbed", TARGET]
candidate_features = [c for c in panel.columns
                       if c not in DROP_COLS
                       and panel[c].dtype != "object"]
print(f"candidate features: {len(candidate_features)}")
print(f"removed (leakage): {LEAKAGE_COLS}")
print(f"new (legit time): mins_on_pitch_so_far")


candidate features: 110
removed (leakage): ['minute_out']
new (legit time): mins_on_pitch_so_far


## Section B - Match-level train/test split

In [7]:
"""Match-level holdout (matches notebook 02 split)."""
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(panel, panel[TARGET], groups=panel[GROUP]))
train = panel.iloc[train_idx].reset_index(drop=True)
test = panel.iloc[test_idx].reset_index(drop=True)

X_train = train[candidate_features].fillna(0.0)
y_train = train[TARGET].astype(int)
g_train = train[GROUP].values
X_test = test[candidate_features].fillna(0.0)
y_test = test[TARGET].astype(int)
cp_train = train["checkpoint"].values
cp_test = test["checkpoint"].values

n_pos, n_neg = int(y_train.sum()), int((1 - y_train).sum())
spw = n_neg / max(n_pos, 1)
print(f"train: {X_train.shape}, fixtures={train[GROUP].nunique()}, "
      f"pos={n_pos}, neg={n_neg}, spw={spw:.2f}")
print(f"test:  {X_test.shape}, fixtures={test[GROUP].nunique()}, "
      f"pos={int(y_test.sum())}")


train: (2394, 110), fixtures=24, pos=173, neg=2221, spw=12.84
test:  (720, 110), fixtures=7, pos=29


## Section C - Variance + correlation pre-filter

Drop near-constant features and one of any pair with `|rho| ≥ 0.95`.
This is a *coarse* pre-filter before the actual selection methods - we
only remove features that are mathematically duplicated.


In [8]:
"""Variance filter."""
def is_near_constant(s, threshold=0.99):
    if s.dropna().nunique() <= 1: return True
    return s.value_counts(normalize=True, dropna=False).iloc[0] >= threshold

near_const = [c for c in candidate_features if is_near_constant(X_train[c])]
print(f"near-constant: {len(near_const)}")
for c in near_const: print(f"  {c}")

features_var = [c for c in candidate_features if c not in near_const]
print(f"after variance filter: {len(features_var)}")


near-constant: 7
  is_penalty_taker
  top_third_run_share_G
  last15_sprints_G
  press_turnover_rate_G
  forward_pass_share_G
  cumul_passes_G
  passes_received_share_G
after variance filter: 103


In [9]:
"""Coarse correlation pre-prune at rho=0.95.

Order by |Pearson r| with target, keep highest-r feature in each pair.
"""
y = y_train.values
r = pd.Series({
    c: abs(np.corrcoef(X_train[c], y)[0, 1]) if X_train[c].std() > 0 else 0.0
    for c in features_var
})
ordered = r.sort_values(ascending=False).index.tolist()

corr_abs = X_train[features_var].corr().abs()
kept = []
dropped_pairs = []
for f in ordered:
    bad = False
    for k in kept:
        if corr_abs.at[f, k] >= 0.95:
            dropped_pairs.append((f, k, float(corr_abs.at[f, k])))
            bad = True; break
    if not bad: kept.append(f)
print(f"after rho=0.95 prune: {len(kept)}")
print(f"dropped pairs: {len(dropped_pairs)}")
for d, k, rho in dropped_pairs[:10]:
    print(f"  {d:55s} <- {k:40s} rho={rho:.3f}")

features_pre = kept
X_train_pre = X_train[features_pre]
X_test_pre = X_test[features_pre]


after rho=0.95 prune: 91
dropped pairs: 12
  passes_received_share_A                                 <- pos_A                                    rho=0.958
  ratio_mean_max_speed                                    <- ratio_peak_speed                         rho=0.985
  mins_on_pitch_so_far                                    <- ratio_peak_speed                         rho=0.977
  minutes_active                                          <- ratio_peak_speed                         rho=0.977
  active_windows                                          <- ratio_peak_speed                         rho=0.977
  pos_D                                                   <- passes_received_share_D                  rho=0.991
  last15_shots                                            <- last15_shots_top_third                   rho=0.979
  last15_intensity                                        <- last15_shots_top_third                   rho=0.979
  intensity_uplift                                        <- 

## Section D - LASSO with C-sweep

L1-regularised logistic regression at multiple regularisation strengths.
Pick the C that maximises CV AUC. Features with non-zero coefs at that
C are the LASSO selection.


In [10]:
"""LASSO C-sweep."""
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train_pre)
X_test_scaled = sc.transform(X_test_pre)

C_values = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
sweep_rows = []
for C in C_values:
    aucs = []
    for tr, va in GroupKFold(5).split(X_train_scaled, y_train, g_train):
        clf = LogisticRegression(
            penalty="l1", solver="liblinear", C=C,
            class_weight="balanced", max_iter=2000, random_state=RANDOM_SEED,
        )
        clf.fit(X_train_scaled[tr], y_train.iloc[tr])
        proba = clf.predict_proba(X_train_scaled[va])[:, 1]
        aucs.append(roc_auc_score(y_train.iloc[va], proba))
    nonzero = (np.abs(clf.coef_[0]) > 1e-8).sum()
    sweep_rows.append({"C": C, "cv_auc_mean": np.mean(aucs), "n_features": int(nonzero)})

sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df.to_string(index=False))
best_C = float(sweep_df.loc[sweep_df["cv_auc_mean"].idxmax(), "C"])
print(f"\nbest C: {best_C}")


   C  cv_auc_mean  n_features
0.05       0.5819          40
0.10       0.5628          54
0.20       0.5465          67
0.50       0.5390          75
1.00       0.5394          78
2.00       0.5386          82

best C: 0.05


In [11]:
"""LASSO selection at best C (refit on full training set)."""
clf_lasso = LogisticRegression(
    penalty="l1", solver="liblinear", C=best_C,
    class_weight="balanced", max_iter=2000, random_state=RANDOM_SEED,
)
clf_lasso.fit(X_train_scaled, y_train)
lasso_features = pd.Series(np.abs(clf_lasso.coef_[0]), index=features_pre)
lasso_features = lasso_features[lasso_features > 1e-8].sort_values(ascending=False)
print(f"LASSO selected {len(lasso_features)} features:")
print(lasso_features.head(25).round(4).to_string())
selection_lasso = lasso_features.index.tolist()


LASSO selected 42 features:
pos_A                                           0.3732
formation_offensiveness                         0.3175
fatigue_gap                                     0.3040
match_minute_at_cp                              0.2554
ratio_distance                                  0.2251
last15_sprints_M                                0.2102
top_third_press_share_x_set_piece_share         0.2030
top_third_pass_x_top_third_run                  0.1458
is_home_int                                     0.1453
player_shot_share                               0.1436
share_right_foot                                0.1399
jersey_number                                   0.1296
cumul_mean_max_speed                            0.1250
subbed_int                                      0.1159
composure_under_pressure_ratio                  0.1084
forward_pass_share_D                            0.1052
cumul_passes_A                                  0.1044
top_third_run_share_M                

## Section E - Stability LASSO (bootstrap)

50 bootstrap samples (sampling rows with replacement). Fit LASSO on each.
Keep features picked in ≥60% of bootstraps.


In [12]:
"""Stability LASSO."""
N_BOOTSTRAPS = 50
bootstrap_picks = pd.DataFrame(0, index=features_pre, columns=range(N_BOOTSTRAPS))
rng = np.random.RandomState(RANDOM_SEED)

for b in range(N_BOOTSTRAPS):
    idx_boot = rng.choice(len(X_train_scaled), size=len(X_train_scaled), replace=True)
    Xb = X_train_scaled[idx_boot]
    yb = y_train.iloc[idx_boot].reset_index(drop=True)
    clf = LogisticRegression(
        penalty="l1", solver="liblinear", C=best_C,
        class_weight="balanced", max_iter=2000, random_state=b,
    )
    clf.fit(Xb, yb)
    bootstrap_picks.iloc[:, b] = (np.abs(clf.coef_[0]) > 1e-8).astype(int)

stability_score = bootstrap_picks.mean(axis=1).sort_values(ascending=False)
print("top 25 features by stability score (% bootstraps selected):")
print((stability_score.head(25) * 100).round(1).to_string())

THRESHOLD_STABILITY = 0.60
selection_stability = stability_score[stability_score >= THRESHOLD_STABILITY].index.tolist()
print(f"\nstability LASSO @ threshold {THRESHOLD_STABILITY}: {len(selection_stability)} features")


top 25 features by stability score (% bootstraps selected):
fatigue_gap                                100.0
formation_offensiveness                    100.0
composure_under_pressure_ratio              98.0
top_third_press_share_x_set_piece_share     94.0
is_home_int                                 94.0
top_third_pass_x_top_third_run              94.0
match_minute_at_cp                          92.0
last15_sprints_M                            92.0
ratio_distance                              92.0
pos_A                                       90.0
jersey_number                               86.0
share_right_foot                            86.0
shot_intent_under_fatigue                   84.0
cumul_mean_max_speed                        82.0
subbed_int                                  82.0
subbed_x_last15_intensity_shots             80.0
player_shot_share                           78.0
cumul_passes_A                              76.0
mean_abs_pass_angle                         74.0
forward_p

## Section F - BorutaPy (XGBoost + shadow features)

Boruta creates "shadow" features by shuffling the originals and tests
whether real features beat their shadows in tree importance. Pure
algorithmic decision — no thresholds to tune.

We use XGBoost as the base estimator (matches the downstream model).


In [13]:
"""BorutaPy with XGBoost."""
xgb_for_boruta = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
)
boruta_selector = BorutaPy(
    xgb_for_boruta, n_estimators="auto",
    max_iter=50, random_state=RANDOM_SEED, verbose=0,
)
# BorutaPy requires numpy arrays, not DataFrames.
boruta_selector.fit(X_train_pre.values, y_train.values)
confirmed = [features_pre[i] for i in range(len(features_pre)) if boruta_selector.support_[i]]
tentative = [features_pre[i] for i in range(len(features_pre)) if boruta_selector.support_weak_[i]]
print(f"Boruta confirmed:  {len(confirmed)}")
for f in confirmed: print(f"  {f}")
print(f"\nBoruta tentative:  {len(tentative)}")
selection_boruta = confirmed + tentative   # use both for breadth


Boruta confirmed:  3
  pos_A
  forward_pass_share_D
  cumul_sprints

Boruta tentative:  3


## Section G - RFE-CV

Recursive Feature Elimination with cross-validation. Fits XGBoost
iteratively, removes least-important features, picks the count that
maximises CV AUC.


In [14]:
"""RFE-CV with XGBoost (lighter config for speed)."""
xgb_for_rfe = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    scale_pos_weight=spw, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
)
rfe = RFECV(
    estimator=xgb_for_rfe,
    step=2, min_features_to_select=5,
    cv=GroupKFold(5).split(X_train_pre, y_train, g_train),
    scoring="roc_auc", n_jobs=1,
)
rfe.fit(X_train_pre, y_train)
selection_rfe = [features_pre[i] for i in range(len(features_pre)) if rfe.support_[i]]
print(f"RFE-CV selected {rfe.n_features_} features:")
for f in selection_rfe: print(f"  {f}")


RFE-CV selected 5 features:
  top_third_pass_share
  forward_pass_share_D
  cumul_sprints
  jersey_number
  cumul_passes_M


## Section H - Consensus shortlist

Each feature gets one vote per method that includes it. Features with
**≥2 votes** form the consensus shortlist. The consensus list is
prefereed for the modelling step because it survives multiple
selection paradigms (linear vs tree, single vs bootstrap).


In [15]:
"""Vote tally."""
votes = pd.DataFrame(0, index=features_pre,
                     columns=["lasso", "stability", "boruta", "rfe"])
votes.loc[selection_lasso, "lasso"] = 1
votes.loc[selection_stability, "stability"] = 1
votes.loc[selection_boruta, "boruta"] = 1
votes.loc[selection_rfe, "rfe"] = 1
votes["total"] = votes.sum(axis=1)
votes_sorted = votes.sort_values("total", ascending=False)
print("vote distribution:")
print(votes_sorted["total"].value_counts().sort_index(ascending=False).to_string())
print()
print("top 30 features by votes:")
print(votes_sorted.head(30).to_string())

selection_consensus_strong = votes[votes["total"] >= 3].index.tolist()
selection_consensus_soft = votes[votes["total"] >= 2].index.tolist()
print(f"\nstrong consensus (>=3 votes): {len(selection_consensus_strong)}")
print(f"soft consensus (>=2 votes): {len(selection_consensus_soft)}")


vote distribution:
total
4     1
3     4
2    29
1    13
0    44

top 30 features by votes:
                                              lasso  stability  boruta  rfe  total
forward_pass_share_D                              1          1       1    1      4
pos_A                                             1          1       1    0      3
cumul_sprints                                     1          0       1    1      3
jersey_number                                     1          1       0    1      3
cumul_passes_M                                    1          1       0    1      3
cumul_passes                                      1          1       0    0      2
top_third_press_share_above_team_avg              1          1       0    0      2
mean_abs_pass_angle                               1          1       0    0      2
fatigue_gap                                       1          1       0    0      2
forward_pass_share                                1          1       0    0   

## Section I - Train XGBoost on each shortlist

For each candidate feature subset (LASSO, Stability, Boruta, RFE,
Consensus-strong, Consensus-soft, Full pre-filter), train XGBoost with
GroupKFold OOF and measure performance.


In [16]:
"""OOF helper."""
def xgb_factory(spw_):
    return lambda: xgb.XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=3,
        min_child_weight=10, subsample=0.9, colsample_bytree=0.85,
        scale_pos_weight=spw_,
        objective="binary:logistic", eval_metric="auc",
        early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
        tree_method="hist",
    )


def cv_oof(X, y, groups, factory, n_splits=5):
    oof = np.zeros(len(X))
    best_iters = []
    for tr, va in GroupKFold(n_splits).split(X, y, groups):
        m = factory()
        m.fit(X.iloc[tr], y.iloc[tr],
              eval_set=[(X.iloc[va], y.iloc[va])], verbose=False)
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        best_iters.append(m.best_iteration)
    return oof, best_iters


SUBSETS = {
    "lasso": selection_lasso,
    "stability": selection_stability,
    "boruta": selection_boruta,
    "rfe": selection_rfe,
    "consensus_3+": selection_consensus_strong,
    "consensus_2+": selection_consensus_soft,
    "full_prefilter": features_pre,
}
factory = xgb_factory(spw)

results = []
oof_per_subset = {}
for name, feats in SUBSETS.items():
    if len(feats) == 0:
        print(f"{name}: 0 features -> skip"); continue
    oof, biters = cv_oof(X_train_pre[feats], y_train, g_train, factory)
    auc = roc_auc_score(y_train, oof)
    ap = average_precision_score(y_train, oof)
    results.append({
        "subset": name, "n_features": len(feats),
        "oof_auc": auc, "oof_ap": ap,
        "median_n_estimators": int(np.median(biters)),
    })
    oof_per_subset[name] = oof
    print(f"  {name:15s}  k={len(feats):3d}  OOF AUC={auc:.4f}  AP={ap:.4f}")

results_df = pd.DataFrame(results).sort_values("oof_auc", ascending=False).reset_index(drop=True)
print()
print(results_df.to_string(index=False))


  lasso            k= 42  OOF AUC=0.5897  AP=0.1039
  stability        k= 34  OOF AUC=0.6085  AP=0.1332
  boruta           k=  6  OOF AUC=0.5979  AP=0.0961
  rfe              k=  5  OOF AUC=0.6578  AP=0.1162
  consensus_3+     k=  5  OOF AUC=0.6360  AP=0.1334
  consensus_2+     k= 34  OOF AUC=0.6132  AP=0.1007
  full_prefilter   k= 91  OOF AUC=0.5403  AP=0.0811

        subset  n_features  oof_auc  oof_ap  median_n_estimators
           rfe           5   0.6578  0.1162                   24
  consensus_3+           5   0.6360  0.1334                    9
  consensus_2+          34   0.6132  0.1007                    7
     stability          34   0.6085  0.1332                    4
        boruta           6   0.5979  0.0961                    3
         lasso          42   0.5897  0.1039                   21
full_prefilter          91   0.5403  0.0811                    3


In [17]:
"""Pick winning subset by OOF AUC."""
winner = results_df.iloc[0]
WINNER_NAME = winner["subset"]
WINNER_FEATURES = SUBSETS[WINNER_NAME]
print(f"WINNER: {WINNER_NAME}")
print(f"  features: {len(WINNER_FEATURES)}")
print(f"  OOF AUC: {winner['oof_auc']:.4f}")
print(f"  features list:")
for f in WINNER_FEATURES: print(f"    {f}")


WINNER: rfe
  features: 5
  OOF AUC: 0.6578
  features list:
    top_third_pass_share
    forward_pass_share_D
    cumul_sprints
    jersey_number
    cumul_passes_M


## Section J - Per-checkpoint isotonic calibration

Each checkpoint has a fundamentally different base rate (9.85% at H1_15,
4.10% at H2_30). One global calibrator distorts predictions per cp.
We fit a separate isotonic regression per cp **bucket** (H2_45 + ET1_15
pooled with H2_30 due to small n).


In [18]:
"""Define cp buckets."""
def cp_to_bucket(cp):
    if cp in ("H2_30", "H2_45", "ET1_15"):
        return "H2_30+"
    return cp

bucket_train = pd.Series(cp_train).map(cp_to_bucket).values
bucket_test = pd.Series(cp_test).map(cp_to_bucket).values
print("bucket distribution (train):")
print(pd.Series(bucket_train).value_counts().sort_index().to_string())
print("\nbucket distribution (test):")
print(pd.Series(bucket_test).value_counts().sort_index().to_string())


bucket distribution (train):
H1_15     479
H1_30     475
H1_45     475
H2_15     476
H2_30+    489

bucket distribution (test):
H1_15     140
H1_30     140
H1_45     140
H2_15     140
H2_30+    160


In [19]:
"""Refit final XGBoost on full training set."""
median_n_est = int(winner["median_n_estimators"]) + 10
final_xgb = xgb.XGBClassifier(
    n_estimators=median_n_est, learning_rate=0.05, max_depth=3,
    min_child_weight=10, subsample=0.9, colsample_bytree=0.85,
    scale_pos_weight=spw,
    objective="binary:logistic", eval_metric="auc",
    n_jobs=-1, random_state=RANDOM_SEED, tree_method="hist",
)
final_xgb.fit(X_train_pre[WINNER_FEATURES], y_train, verbose=False)
test_proba_raw = final_xgb.predict_proba(X_test_pre[WINNER_FEATURES])[:, 1]
oof_raw = oof_per_subset[WINNER_NAME]

# Test-set raw metrics
print(f"=== TEST raw (no calibration) ===")
print(f"  AUC:   {roc_auc_score(y_test, test_proba_raw):.4f}")
print(f"  AP:    {average_precision_score(y_test, test_proba_raw):.4f}")
print(f"  Brier: {brier_score_loss(y_test, test_proba_raw):.4f}")


=== TEST raw (no calibration) ===
  AUC:   0.8122
  AP:    0.1165
  Brier: 0.2056


In [20]:
"""Strategy 1: Global isotonic calibration."""
iso_global = IsotonicRegression(out_of_bounds="clip")
iso_global.fit(oof_raw, y_train)
oof_cal_global = iso_global.transform(oof_raw)
test_cal_global = iso_global.transform(test_proba_raw)

print(f"=== TEST global isotonic ===")
print(f"  AUC:   {roc_auc_score(y_test, test_cal_global):.4f}")
print(f"  AP:    {average_precision_score(y_test, test_cal_global):.4f}")
print(f"  Brier: {brier_score_loss(y_test, test_cal_global):.4f}")


=== TEST global isotonic ===
  AUC:   0.8102
  AP:    0.1099
  Brier: 0.0386


In [21]:
"""Strategy 2: Per-checkpoint isotonic calibration."""
test_cal_percp = test_proba_raw.copy()
oof_cal_percp = oof_raw.copy()
calibrators = {}
for bucket in pd.unique(bucket_train):
    mask_tr = bucket_train == bucket
    mask_te = bucket_test == bucket
    if mask_tr.sum() < 10:  # too few to calibrate
        calibrators[bucket] = iso_global
        continue
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_raw[mask_tr], y_train.iloc[mask_tr])
    calibrators[bucket] = iso
    oof_cal_percp[mask_tr] = iso.transform(oof_raw[mask_tr])
    if mask_te.sum() > 0:
        test_cal_percp[mask_te] = iso.transform(test_proba_raw[mask_te])

print(f"=== TEST per-checkpoint isotonic ===")
print(f"  AUC:   {roc_auc_score(y_test, test_cal_percp):.4f}")
print(f"  AP:    {average_precision_score(y_test, test_cal_percp):.4f}")
print(f"  Brier: {brier_score_loss(y_test, test_cal_percp):.4f}")


=== TEST per-checkpoint isotonic ===
  AUC:   0.7998
  AP:    0.1221
  Brier: 0.0394


In [22]:
"""Per-bucket Brier comparison (calibration quality)."""
rows = []
for bucket in sorted(pd.unique(bucket_test)):
    mask = bucket_test == bucket
    if mask.sum() == 0: continue
    rows.append({
        "bucket": bucket, "n_test": int(mask.sum()),
        "actual_rate": float(y_test.iloc[mask].mean()),
        "raw_mean_prob": float(test_proba_raw[mask].mean()),
        "cal_global_prob": float(test_cal_global[mask].mean()),
        "cal_percp_prob": float(test_cal_percp[mask].mean()),
        "brier_raw": brier_score_loss(y_test.iloc[mask], test_proba_raw[mask]),
        "brier_global": brier_score_loss(y_test.iloc[mask], test_cal_global[mask]),
        "brier_percp": brier_score_loss(y_test.iloc[mask], test_cal_percp[mask]),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))


bucket  n_test  actual_rate  raw_mean_prob  cal_global_prob  cal_percp_prob  brier_raw  brier_global  brier_percp
 H1_15     140       0.0714         0.4891           0.0960          0.1158     0.2420        0.0631       0.0650
 H1_30     140       0.0571         0.4519           0.0799          0.0891     0.2171        0.0520       0.0527
 H1_45     140       0.0500         0.4353           0.0761          0.0745     0.2061        0.0449       0.0450
 H2_15     140       0.0214         0.3905           0.0654          0.0469     0.1790        0.0231       0.0214
H2_30+     160       0.0062         0.3895           0.0689          0.0584     0.1867        0.0136       0.0162


## Section K - Per-checkpoint threshold tuning

For each cp bucket, find the threshold maximising **per-cp balanced
accuracy** (using OOF predictions, training-only).


In [23]:
"""Per-bucket threshold sweep on OOF."""
THRESH_RANGE = np.linspace(0.01, 0.6, 60)

def best_threshold(probs, y, thresholds=THRESH_RANGE):
    bas = [balanced_accuracy_score(y, (probs >= t).astype(int)) for t in thresholds]
    i = int(np.argmax(bas))
    return float(thresholds[i]), float(bas[i])

# Single global threshold (baseline).
global_thr, global_oof_ba = best_threshold(oof_cal_percp, y_train)
print(f"global threshold: {global_thr:.3f}  -> OOF BA = {global_oof_ba:.4f}")
print()

# Per-bucket thresholds.
percp_thresholds = {}
print("per-cp thresholds (from OOF):")
for bucket in sorted(pd.unique(bucket_train)):
    mask = bucket_train == bucket
    if mask.sum() < 50:
        percp_thresholds[bucket] = global_thr
        continue
    thr, ba = best_threshold(oof_cal_percp[mask], y_train.iloc[mask])
    percp_thresholds[bucket] = thr
    print(f"  {bucket:8s}  n={int(mask.sum()):4d}  thr={thr:.3f}  OOF BA={ba:.4f}")


global threshold: 0.070  -> OOF BA = 0.6556

per-cp thresholds (from OOF):
  H1_15     n= 479  thr=0.060  OOF BA=0.6480
  H1_30     n= 475  thr=0.070  OOF BA=0.6457
  H1_45     n= 475  thr=0.060  OOF BA=0.5942
  H2_15     n= 476  thr=0.040  OOF BA=0.6574
  H2_30+    n= 489  thr=0.050  OOF BA=0.6671


In [24]:
"""Apply per-cp thresholds to test set."""
test_pred_global = (test_cal_percp >= global_thr).astype(int)
test_pred_percp = np.zeros(len(test_cal_percp), dtype=int)
for bucket, thr in percp_thresholds.items():
    mask = bucket_test == bucket
    test_pred_percp[mask] = (test_cal_percp[mask] >= thr).astype(int)

ba_global = balanced_accuracy_score(y_test, test_pred_global)
ba_percp = balanced_accuracy_score(y_test, test_pred_percp)
print(f"TEST balanced accuracy:")
print(f"  global threshold ({global_thr:.3f}):  {ba_global:.4f}")
print(f"  per-cp thresholds:               {ba_percp:.4f}")
print(f"  delta:                            {ba_percp - ba_global:+.4f}")


TEST balanced accuracy:
  global threshold (0.070):  0.7062
  per-cp thresholds:               0.6983
  delta:                            -0.0080


In [25]:
"""Per-bucket BA in the test set."""
rows = []
for bucket in sorted(pd.unique(bucket_test)):
    mask = bucket_test == bucket
    if mask.sum() == 0: continue
    if int(y_test.iloc[mask].sum()) == 0 or int((1 - y_test.iloc[mask]).sum()) == 0:
        ba_g = ba_p = float("nan")
    else:
        ba_g = balanced_accuracy_score(y_test.iloc[mask], test_pred_global[mask])
        ba_p = balanced_accuracy_score(y_test.iloc[mask], test_pred_percp[mask])
    rows.append({"bucket": bucket, "n": int(mask.sum()),
                 "n_pos": int(y_test.iloc[mask].sum()),
                 "BA_global": ba_g, "BA_percp": ba_p,
                 "thr_percp": percp_thresholds.get(bucket, global_thr)})
print(pd.DataFrame(rows).round(4).to_string(index=False))


bucket   n  n_pos  BA_global  BA_percp  thr_percp
 H1_15 140     10     0.6769    0.6769       0.06
 H1_30 140      8     0.6667    0.6667       0.07
 H1_45 140      7     0.7406    0.7406       0.06
 H2_15 140      3     0.6873    0.6800       0.04
H2_30+ 160      1     0.3522    0.3239       0.05


## Section L - Final test-set evaluation summary

Aggregate all combinations on the held-out test fixtures.


In [26]:
"""Final summary."""
final_rows = []
for cal_name, test_probs in [
    ("raw", test_proba_raw),
    ("global_iso", test_cal_global),
    ("percp_iso", test_cal_percp),
]:
    for thr_name, thr_strategy in [("global_thr", "global"), ("percp_thr", "percp")]:
        if thr_strategy == "global":
            thr_g, _ = best_threshold(
                iso_global.transform(oof_raw) if cal_name == "global_iso"
                else (oof_cal_percp if cal_name == "percp_iso" else oof_raw),
                y_train,
            )
            preds = (test_probs >= thr_g).astype(int)
        else:
            preds = np.zeros(len(test_probs), dtype=int)
            for bucket in pd.unique(bucket_test):
                mask = bucket_test == bucket
                preds[mask] = (test_probs[mask] >= percp_thresholds.get(bucket, global_thr)).astype(int)
        final_rows.append({
            "calibration": cal_name, "threshold": thr_name,
            "AUC": roc_auc_score(y_test, test_probs),
            "AP": average_precision_score(y_test, test_probs),
            "Brier": brier_score_loss(y_test, test_probs),
            "BA": balanced_accuracy_score(y_test, preds),
        })

final_df = pd.DataFrame(final_rows).round(4)
print(final_df.to_string(index=False))
print()
best = final_df.loc[final_df["BA"].idxmax()]
print(f"BEST BY BA: {best['calibration']} + {best['threshold']}  ->  BA={best['BA']:.4f}, AUC={best['AUC']:.4f}")


calibration  threshold    AUC     AP  Brier     BA
        raw global_thr 0.8122 0.1165 0.2056 0.6990
        raw  percp_thr 0.8122 0.1165 0.2056 0.5000
 global_iso global_thr 0.8102 0.1099 0.0386 0.7148
 global_iso  percp_thr 0.8102 0.1099 0.0386 0.7024
  percp_iso global_thr 0.7998 0.1221 0.0394 0.7062
  percp_iso  percp_thr 0.7998 0.1221 0.0394 0.6983

BEST BY BA: global_iso + global_thr  ->  BA=0.7148, AUC=0.8102


In [29]:
"""Detailed test-set classification statistics."""
def classification_stats(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    npv = tn / (tn + fn) if (tn + fn) else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "npv": npv,
        "f1": f1,
        "auc": roc_auc_score(y_true, y_prob),
        "ap": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "logloss": log_loss(y_true, np.clip(y_prob, 1e-8, 1 - 1e-8)),
    }


detailed_rows = []
for cal_name, test_probs in [
    ("raw", test_proba_raw),
    ("global_iso", test_cal_global),
    ("percp_iso", test_cal_percp),
]:
    for thr_name, thr_strategy in [("global_thr", "global"), ("percp_thr", "percp")]:
        if thr_strategy == "global":
            thr_used, _ = best_threshold(
                iso_global.transform(oof_raw) if cal_name == "global_iso"
                else (oof_cal_percp if cal_name == "percp_iso" else oof_raw),
                y_train,
            )
            preds = (test_probs >= thr_used).astype(int)
        else:
            thr_used = np.nan
            preds = np.zeros(len(test_probs), dtype=int)
            for bucket in pd.unique(bucket_test):
                mask = bucket_test == bucket
                preds[mask] = (test_probs[mask] >= percp_thresholds.get(bucket, global_thr)).astype(int)

        row = {
            "calibration": cal_name,
            "threshold": thr_name,
            "thr_used": thr_used,
        }
        row.update(classification_stats(y_test, preds, test_probs))
        detailed_rows.append(row)

detailed_final_df = pd.DataFrame(detailed_rows).round(4)
print(detailed_final_df.to_string(index=False))

best_by_f1 = detailed_final_df.loc[detailed_final_df["f1"].idxmax()]
best_by_recall = detailed_final_df.loc[detailed_final_df["recall"].idxmax()]
best_by_precision = detailed_final_df.loc[detailed_final_df["precision"].idxmax()]

print("\nBest by F1:")
print(best_by_f1.to_string())

print("\nBest by recall:")
print(best_by_recall.to_string())

print("\nBest by precision:")
print(best_by_precision.to_string())

calibration  threshold  thr_used  tn  fp  fn  tp  accuracy  balanced_accuracy  precision  recall  specificity    npv     f1    auc     ap  brier  logloss
        raw global_thr      0.51 418 273   6  23    0.6125             0.6990     0.0777  0.7931       0.6049 0.9858 0.1415 0.8122 0.1165 0.2056   0.5898
        raw  percp_thr       NaN   0 691   0  29    0.0403             0.5000     0.0403  1.0000       0.0000 0.0000 0.0774 0.8122 0.1165 0.2056   0.5898
 global_iso global_thr      0.05 416 275   5  24    0.6111             0.7148     0.0803  0.8276       0.6020 0.9881 0.1463 0.8102 0.1099 0.0386   0.1616
 global_iso  percp_thr       NaN 375 316   4  25    0.5556             0.7024     0.0733  0.8621       0.5427 0.9894 0.1351 0.8102 0.1099 0.0386   0.1616
  percp_iso global_thr      0.07 428 263   6  23    0.6264             0.7062     0.0804  0.7931       0.6194 0.9862 0.1460 0.7998 0.1221 0.0394   0.1599
  percp_iso  percp_thr       NaN 417 274   6  23    0.6111             0.698

## Section M - Persist artefacts

In [30]:
"""Save."""
import json

# Model + features.
with open(ART_DIR / "xgb_model.pkl", "wb") as f:
    pickle.dump(final_xgb, f)
with open(ART_DIR / "calibrators.pkl", "wb") as f:
    pickle.dump(calibrators, f)
with open(ART_DIR / "global_calibrator.pkl", "wb") as f:
    pickle.dump(iso_global, f)

# Raw matrices and predictions for downstream XAI.
X_train_pre[WINNER_FEATURES].to_csv(ART_DIR / "X_train_raw.csv", index=False)
X_test_pre[WINNER_FEATURES].to_csv(ART_DIR / "X_test_raw.csv", index=False)
y_train.to_csv(ART_DIR / "y_train_raw.csv", index=False, header=True)
y_test.to_csv(ART_DIR / "y_test_raw.csv", index=False, header=True)

oof_df = train[["player_appearance_id", "checkpoint", "fixture_id", "scored_after"]].copy()
oof_df["oof_raw"] = oof_raw
oof_df["oof_cal_global"] = oof_cal_global
oof_df["oof_cal_percp"] = oof_cal_percp
oof_df.to_csv(ART_DIR / "oof_predictions.csv", index=False)

test_df = test[["player_appearance_id", "checkpoint", "fixture_id", "scored_after"]].copy()
test_df["test_raw"] = test_proba_raw
test_df["test_cal_global"] = test_cal_global
test_df["test_cal_percp"] = test_cal_percp
test_df["test_pred_percp"] = test_pred_percp
test_df.to_csv(ART_DIR / "test_predictions.csv", index=False)

# Selection comparison.
results_df.to_csv(ART_DIR / "selection_comparison.csv", index=False)
votes_sorted.to_csv(ART_DIR / "feature_votes.csv")

# Config.
config = {
    "winner_subset": WINNER_NAME,
    "winner_features": WINNER_FEATURES,
    "n_features": len(WINNER_FEATURES),
    "best_C_lasso": best_C,
    "n_estimators": median_n_est,
    "scale_pos_weight": float(spw),
    "global_threshold": global_thr,
    "percp_thresholds": {k: float(v) for k, v in percp_thresholds.items()},
    "test_metrics": final_df.to_dict(orient="records"),
}
with open(ART_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"saved to {ART_DIR}/:")
for p in sorted(ART_DIR.iterdir()):
    print(f"  {p.name:30s} {p.stat().st_size / 1024:.1f} KB")


saved to c:\Users\tymot\projects\wec\models\kitchen_sink/:
  calibrators.pkl                2.2 KB
  config.json                    1.5 KB
  feature_votes.csv              2.9 KB
  global_calibrator.pkl          0.7 KB
  oof_predictions.csv            184.6 KB
  selection_comparison.csv       0.4 KB
  test_predictions.csv           44.5 KB
  X_test_raw.csv                 19.8 KB
  X_train_raw.csv                65.1 KB
  xgb_model.pkl                  40.5 KB
  y_test_raw.csv                 2.1 KB
  y_train_raw.csv                7.0 KB


### Co dalej

* **`03_xai_explanations.ipynb`** — full XAI mirror reference notebooka.
* **`02c_score_state_features.ipynb`** — dodać score state features (current_score_diff,
  team_xg_so_far) i powtórzyć cały selekcyjny pipeline na rozszerzonym poolu.
* **`04_position_stratified.ipynb`** — sub-models per position.
* **`05_stacking_ensemble.ipynb`** — final boost.
